# 01 - Capa Bronze\n\nIngesta del archivo `venta_tiendas.csv` desde Google Drive y escritura en Delta Lake.\n\n**Principio Medallion:** Bronze conserva los datos tal como llegan desde la fuente, incluyendo `fecha_transaccion` con `/`.

# Configuración base

Este notebook está preparado para ejecutarse en Google Colab usando Google Drive personal.

Estructura esperada en Drive:

```text
Proyecto_BigData_Forus/
├── raw/
│   └── venta_tiendas.csv
├── bronze/
├── silver/
├── gold/
└── evidencias/
```


In [11]:
# Instalación de dependencias para Colab
!pip install -q pyspark==3.5.1 delta-spark==3.2.0


In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Corrected RUTA_BASE to be a directory path for the project root
# Assuming 'Proyecto_BigData_Forus' is the intended project folder in MyDrive
RUTA_BASE = "/content/drive/MyDrive"

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"
RUTA_EVIDENCIAS = f"{RUTA_BASE}/evidencias"

ARCHIVO_VENTAS = "venta_tiendas.csv"

for ruta in [RUTA_RAW, RUTA_BRONZE, RUTA_SILVER, RUTA_GOLD, RUTA_EVIDENCIAS]:
    os.makedirs(ruta, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("Forus_Fase2_BigData")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("Ruta base:", RUTA_BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark: 3.5.1
Ruta base: /content/drive/MyDrive


## 1. Lectura RAW desde Drive

In [14]:
ruta_csv = '/content/drive/MyDrive/venta_tiendas.csv'

inicio = time.time()

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)  # Bronze preserva valores originales como texto
    .option("encoding", "UTF-8")
    .csv(ruta_csv)
)

tiempo_lectura = time.time() - inicio

print("Registros RAW:", df_raw.count())
print("Tiempo lectura RAW:", round(tiempo_lectura, 2), "segundos")
df_raw.printSchema()
df_raw.show(5, truncate=False)

Registros RAW: 2250970
Tiempo lectura RAW: 5.84 segundos
root
 |-- id_canal: string (nullable = true)
 |-- numero_transaccion: string (nullable = true)
 |-- numero_pos: string (nullable = true)
 |-- numero_boleta: string (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: string (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- unidades: string (nullable = true)
 |-- venta: string (nullable = true)
 |-- costo: string (nullable = true)

+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_boleta|fecha_transaccion        |cod_tienda_facturacion|tipo_documento|id_producto|unidades|venta|costo|
+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----

In [15]:
import os

print(f"Contenido de la carpeta RAW ({RUTA_RAW}):")
for item in os.listdir(RUTA_RAW):
    print(item)


Contenido de la carpeta RAW (/content/drive/MyDrive/raw):
venta_tiendas.csv


In [16]:
import os

search_file = ARCHIVO_VENTAS # This is 'venta_tiendas.csv'

print(f"Buscando '{search_file}' en tu Google Drive...")

# Use a shell command to find the file in the mounted drive
# This might take a while depending on the size of your drive
find_command = f"find /content/drive -name '{search_file}'"
os.system(find_command)

print("Búsqueda finalizada. Si el archivo fue encontrado, su ruta se imprimió arriba.")

Buscando 'venta_tiendas.csv' en tu Google Drive...
Búsqueda finalizada. Si el archivo fue encontrado, su ruta se imprimió arriba.


## 2. Escritura Bronze en Delta Lake\n\nNo se transforma `fecha_transaccion` en esta capa.

In [17]:
inicio = time.time()

(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .save(f"{RUTA_BRONZE}/venta_tiendas")
)

tiempo_bronze = time.time() - inicio

print("Bronze escrito correctamente en:", f"{RUTA_BRONZE}/venta_tiendas")
print("Tiempo escritura Bronze:", round(tiempo_bronze, 2), "segundos")


Bronze escrito correctamente en: /content/drive/MyDrive/bronze/venta_tiendas
Tiempo escritura Bronze: 23.53 segundos


## 3. Validación Bronze

In [18]:
df_bronze = spark.read.format("delta").load(f"{RUTA_BRONZE}/venta_tiendas")

print("Registros Bronze:", df_bronze.count())
df_bronze.select("fecha_transaccion").show(5, truncate=False)


Registros Bronze: 2250970
+-------------------------+
|fecha_transaccion        |
+-------------------------+
|14/03/2016 12:00:00 AM CL|
|14/03/2016 12:00:00 AM CL|
|14/03/2016 12:00:00 AM CL|
|14/03/2016 12:00:00 AM CL|
|14/03/2016 12:00:00 AM CL|
+-------------------------+
only showing top 5 rows



## 4. Evidencia de linaje

In [19]:
linaje_bronze = {
    "dataset": "venta_tiendas",
    "origen": f"{RUTA_RAW}/{ARCHIVO_VENTAS}",
    "destino": f"{RUTA_BRONZE}/venta_tiendas",
    "capa": "Bronze",
    "formato": "Delta Lake",
    "transformaciones": "Sin transformaciones; preserva datos originales",
    "fecha_original": "fecha_transaccion conservada con slash y sufijo AM/PM CL"
}

for k, v in linaje_bronze.items():
    print(f"{k}: {v}")


dataset: venta_tiendas
origen: /content/drive/MyDrive/raw/venta_tiendas.csv
destino: /content/drive/MyDrive/bronze/venta_tiendas
capa: Bronze
formato: Delta Lake
transformaciones: Sin transformaciones; preserva datos originales
fecha_original: fecha_transaccion conservada con slash y sufijo AM/PM CL
